In [14]:
import pandas as pd 
import requests
import time
import os
from dotenv import load_dotenv
from tqdm import tqdm

In [21]:
# carregando a chave da API do .env

load_dotenv('../.env')
API_KEY = os.getenv('TMDB_API_KEY')

In [22]:
# carregando base de dados do projeto

df_base = pd.read_csv('../data/base_projeto.csv')

In [23]:
# novas colunas de dados que queremos adicionar no nosso dataset

original_language = []
is_franchise = []
budget = []
revenue = []

In [24]:
# coleta de dados dos tmdb

for imdb_id in tqdm(df_base['tconst']):
    url_tmdb = f"https://api.themoviedb.org/3/movie/{imdb_id}?api_key={API_KEY}"

    try:
        response_tmdb = requests.get(url_tmdb)

        if response_tmdb.status_code == 200:
            data = response_tmdb.json()
            original_language.append(data.get('original_language'))
            budget.append(data.get('budget', 0))
            revenue.append(data.get('revenue', 0))

            if data.get('belongs_to_collection'):
                is_franchise.append(True)
            else:
                is_franchise.append(False)

        else:
            original_language.append(None)
            budget.append(None)
            revenue.append(None)
            is_franchise.append(False)

    except Exception as e: 
        original_language.append(None)
        budget.append(None)
        revenue.append(None)
        is_franchise.append(False)

    time.sleep(0.02)

100%|██████████| 12455/12455 [1:16:43<00:00,  2.71it/s]


In [26]:
# adicionando as colunas no .csv

df_base['original_language'] = original_language
df_base['is_franchise'] = is_franchise
df_base['budget'] = budget
df_base['revenue'] = revenue

df_base.to_csv('../data/df_tmdb.csv', index=False)